In [ ]:
import mdtraj as md
import numpy as np
import pandas as pd
from pathlib import Path
import glob
import os
from collections import defaultdict
import warnings

# Suppress warnings that might clutter the notebook output
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ==============================================================================
# USER CONFIGURABLE PARAMETERS
# ==============================================================================

# --- File Paths and Patterns ---
# Root directory containing the 'repl_*' subfolders with topology and trajectory files.
SEARCH_ROOT_DIR = "."

# Topology and trajectory file naming patterns inside each 'repl' directory.
TOPOLOGY_PATTERN = "state0.{repl}.pdb"      # Placeholder for .format(repl=num)
TRAJECTORY_PATTERN = "center.{repl}.xtc"    # Placeholder for .format(repl=num)

# The Excel output file name pattern. 
OUTPUT_XLSX_PATTERN = "."

# --- Chains and Residues to Evaluate ---
# If None, the code will auto-detect chain 0 and chain 1 residues (using mdtraj indices).
CHAIN_A_RESIDS = None  # Residue indices to evaluate in Chain A
CHAIN_B_RESIDS = None  # Residue indices to evaluate in Chain B

# Specific partner residues to compare against.
# These MUST be the PDB Residue Numbers (resSeq) from your topology.
# Examples of selections: 
#   1. list(range(21, 44)) to select from residue 21 to 43;
#   2. list(range(21, 31)) + [43] to select from residue 21 to 30, plus residue 43;
#   3. [1, 2, 10, 15, 20] to select individual residues

SPECIFIC_PARTNER_RESIDS_B_PDB = list(range(21, 44)) # from key_residues: 22, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38, 39, 42
SPECIFIC_PARTNER_RESIDS_A_PDB = None # For reverse calculation (optional)

# Toggle calculation direction
COMPUTE_A_VS_B = True   # Compute interactions of all A residues vs specific B residues
COMPUTE_B_VS_A = False  # Compute interactions of all B residues vs specific A residues

# --- Analysis Parameters ---
DISTANCE_THRESHOLD_NM = 0.4  # Interaction distance threshold (nm) for occupancy calculation.
USE_HEAVY_ATOMS = True       # True = use non-hydrogen atoms only for distance calculation.
CHUNK_SIZE = 200             # Frames to load per chunk. Set to None to load the whole trajectory.
NUM_TIME_WINDOWS = 3         # Number of windows to split the trajectory into for occupancy calculation.

# --- Output Sheet Names and Filtering Thresholds ---
SHEET_ALL_DATA = 'all_data' # Unfiltered data
SHEET_FILTERED_DATA = 'filtered_data' # Filtered data
OCCUPANCY_FILTER_THRESHOLD = 0.5 # Threshold used for filtering raw data to the 'filtered_data' sheet

# ==============================================================================
# CORE LOGIC
# ==============================================================================

def get_residue_atom_indices(topology, resid_list, only_heavy=True):
    """
    Retrieves the atom indices for a list of residue indices (mdtraj index).
    """
    atom_index_groups = []
    for resid_index in resid_list:
        residue = topology.residue(resid_index)

        if only_heavy:
            atoms = [a.index for a in residue.atoms if a.element.symbol != 'H']
        else:
            atoms = [a.index for a in residue.atoms]

        if len(atoms) == 0:
            # Fallback if no heavy atoms found (e.g. virtual sites), take all
            atoms = [a.index for a in residue.atoms]

        atom_index_groups.append(atoms)
    return atom_index_groups

def find_replica_directories(search_root: str):
    """
    Finds and returns a sorted list of all 'repl_*' subdirectories within the search root.
    It sorts them numerically (repl_2 before repl_10) if the suffix is an integer.
    """
    search_path = Path(search_root)
    if not search_path.exists():
         raise FileNotFoundError(f"Search root directory does not exist: {search_root}")

    # Find directories starting with "repl_"
    replica_dirs = [d for d in search_path.iterdir() if d.is_dir() and d.name.startswith("repl_")]
    
    # Sort helper function to handle "repl_1", "repl_10", etc. correctly
    def sort_key(d):
        try:
            # Extract number after the last underscore
            return int(d.name.split('_')[-1])
        except ValueError:
            # Fallback to string sorting if suffix isn't an integer
            return str(d.name)

    replica_dirs.sort(key=sort_key)

    if not replica_dirs:
        raise FileNotFoundError(f"No 'repl_*' directories found inside {search_root}")
    
    return [str(d) for d in replica_dirs]

def generate_window_masks(total_frames, num_windows):
    """
    Calculates the start/end frames for time windows and generates boolean masks.
    """
    if num_windows < 1:
        return {}
        
    frames_per_window = total_frames // num_windows
    window_definitions = []

    # Define start/end indices for each window
    for window_num in range(num_windows):
        start_idx = window_num * frames_per_window
        # Ensure the last window covers any remainder frames
        if window_num < num_windows - 1:
            end_idx = (window_num + 1) * frames_per_window - 1
        else:
            end_idx = total_frames - 1
        window_definitions.append((start_idx, end_idx))

    window_masks = {}
    for start_idx, end_idx in window_definitions:
        column_name = f"occ_{start_idx+1}_{end_idx+1}"
        mask = np.zeros(total_frames, dtype=bool)
        if end_idx >= start_idx:
            mask[start_idx:end_idx+1] = True
        window_masks[column_name] = mask

    return window_masks

def create_pdb_to_mdtraj_map_robust(topology):
    """
    Creates a dictionary mapping (mdtraj_chain_index, PDB_residue_number)
    to the mdtraj zero-based residue index.
    """
    pdb_to_mdtraj_robust = {}
    for residue in topology.residues:
        pdb_to_mdtraj_robust[(residue.chain.index, residue.resSeq)] = residue.index
    return pdb_to_mdtraj_robust

def run_trajectory_analysis():
    """
    Main execution function. Traverses replica directories, loads trajectories,
    calculates pairwise distances and occupancies, and saves the results.
    """
    try:
        replica_directories = find_replica_directories(SEARCH_ROOT_DIR)
    except FileNotFoundError as e:
        print(f"Global ERROR ❌ - {e}")
        return

    print(f"Global INFO - Found {len(replica_directories)} replica directories. Starting pairwise analysis.")

    # Estimate total chunks for printing (based on first replica)
    estimated_total_chunks = None
    if CHUNK_SIZE is not None and replica_directories:
        try:
            first_replica_dir = Path(replica_directories[0])
            replica_num_approx = first_replica_dir.name.split("_")[-1]
            trajectory_file = first_replica_dir / TRAJECTORY_PATTERN.format(repl=replica_num_approx)

            if trajectory_file.exists():
                t = md.load(str(trajectory_file), frame=0)
                total_frames_approx = t.n_frames
                if t.n_frames > 0: # Avoid division by zero
                    estimated_total_chunks = (total_frames_approx + CHUNK_SIZE - 1) // CHUNK_SIZE
                del t 
        except Exception:
            estimated_total_chunks = None

    for replica_dir_str in replica_directories:
        replica_dir = Path(replica_dir_str)
        try:
            replica_num = replica_dir.name.split("_")[-1]
        except IndexError:
            replica_num = "unknown"

        print(f"\n>>> Starting Interaction Mapping for Replica [{replica_num}] (Directory: {replica_dir})")

        topology_file = replica_dir / TOPOLOGY_PATTERN.format(repl=replica_num)
        trajectory_file = replica_dir / TRAJECTORY_PATTERN.format(repl=replica_num)

        output_xlsx_base = OUTPUT_XLSX_PATTERN.format(repl=replica_num)
        
        if Path(OUTPUT_XLSX_PATTERN).is_absolute():
            output_xlsx_path = output_xlsx_base
        else:
            output_xlsx_path = str(replica_dir / output_xlsx_base)
        
        # Create output directory if it doesn't exist
        Path(output_xlsx_path).parent.mkdir(parents=True, exist_ok=True)
        
        if not topology_file.exists() or not trajectory_file.exists():
            print(f"Replica [{replica_num}] ERROR ❌ - Missing topology or trajectory file. Skipping replica.")
            continue

        try:
            # Load only topology first to check atoms
            topology_traj = md.load(str(topology_file))
        except Exception as e:
            print(f"Replica [{replica_num}] ERROR ❌ - Could not load topology file: {e}. Skipping replica.")
            continue

        chains = list(topology_traj.topology.chains)
        if len(chains) < 2:
            print(f"Replica [{replica_num}] ERROR ❌ - Topology has fewer than 2 chains. Cannot compute interaction. Skipping.")
            continue
            
        chain_a_resids_local = [r.index for r in chains[0].residues] if CHAIN_A_RESIDS is None else list(CHAIN_A_RESIDS)
        chain_b_resids_local = [r.index for r in chains[1].residues] if CHAIN_B_RESIDS is None else list(CHAIN_B_RESIDS)
        pdb_map_robust = create_pdb_to_mdtraj_map_robust(topology_traj.topology)

        specific_b_mdtraj_indices = list(chain_b_resids_local)
        specific_a_mdtraj_indices = list(chain_a_resids_local)

        if SPECIFIC_PARTNER_RESIDS_B_PDB is not None:
            try:
                specific_b_mdtraj_indices = [
                    pdb_map_robust[(1, pdb_res)] for pdb_res in SPECIFIC_PARTNER_RESIDS_B_PDB
                    if (1, pdb_res) in pdb_map_robust
                ]
                if len(specific_b_mdtraj_indices) != len(SPECIFIC_PARTNER_RESIDS_B_PDB):
                     print(f"Replica [{replica_num}] WARNING ⚠️ - Some requested Chain B residues were not found in topology.")
            except KeyError as e:
                print(f"Replica [{replica_num}] ERROR ❌ - PDB residue key error {e}. Skipping replica.")
                continue

        if SPECIFIC_PARTNER_RESIDS_A_PDB is not None:
            try:
                specific_a_mdtraj_indices = [
                    pdb_map_robust[(0, pdb_res)] for pdb_res in SPECIFIC_PARTNER_RESIDS_A_PDB
                    if (0, pdb_res) in pdb_map_robust
                ]
            except KeyError as e:
                print(f"Replica [{replica_num}] ERROR ❌ - PDB residue key error {e}. Skipping replica.")
                continue
        
        chain_a_atom_groups = get_residue_atom_indices(topology_traj.topology, chain_a_resids_local, only_heavy=USE_HEAVY_ATOMS)
        chain_b_atom_groups = get_residue_atom_indices(topology_traj.topology, chain_b_resids_local, only_heavy=USE_HEAVY_ATOMS)

        chain_a_atom_index_map = {resid: atoms for resid, atoms in zip(chain_a_resids_local, chain_a_atom_groups)}
        chain_b_atom_index_map = {resid: atoms for resid, atoms in zip(chain_b_resids_local, chain_b_atom_groups)}

        pairs_to_compute = []
        if COMPUTE_A_VS_B:
            for resid_i in chain_a_resids_local:
                for partner in specific_b_mdtraj_indices:
                    pairs_to_compute.append(('A', resid_i, partner))
        if COMPUTE_B_VS_A:
            for resid_i in chain_b_resids_local:
                for partner in specific_a_mdtraj_indices:
                    pairs_to_compute.append(('B', resid_i, partner))

        distance_accumulator = defaultdict(list)
        
        frame_times_ps_chunks = []
        total_frames = 0

        # Create iterator
        try:
            if CHUNK_SIZE is None:
                iterator = [md.load(str(trajectory_file), top=str(topology_file))]
                chunk_count_str = ""
            else:
                iterator = md.iterload(str(trajectory_file), top=str(topology_file), chunk=CHUNK_SIZE)
                chunk_count_str = f"/{estimated_total_chunks}" if estimated_total_chunks else ""
        except Exception as e:
             print(f"Replica [{replica_num}] ERROR ❌ - Could not load trajectory: {e}. Skipping.")
             continue

        for chunk_index, chunk in enumerate(iterator):
            n_frames_chunk = chunk.n_frames
            total_frames += n_frames_chunk

            print(f"Replica [{replica_num}] - Chunk [{chunk_index+1}{chunk_count_str}]: Processing {n_frames_chunk} frames...", end=' ')

            if hasattr(chunk, 'time') and len(chunk.time) == n_frames_chunk:
                chunk_times_ps = chunk.time.copy()
            else:
                chunk_times_ps = np.arange(total_frames - n_frames_chunk, total_frames, dtype=np.float32)
            
            frame_times_ps_chunks.append(chunk_times_ps.astype(np.float32))

            # --- Optimization: Atom Slicing ---
            # Identify only the atoms needed for this specific chunk of calculation
            used_atom_indices = set()
            if COMPUTE_A_VS_B:
                for resid_i in chain_a_resids_local:
                    used_atom_indices.update(chain_a_atom_index_map[resid_i])
                for r in specific_b_mdtraj_indices:
                    used_atom_indices.update(chain_b_atom_index_map.get(r, []))
            if COMPUTE_B_VS_A:
                for resid_i in chain_b_resids_local:
                    used_atom_indices.update(chain_b_atom_index_map[resid_i])
                for r in specific_a_mdtraj_indices:
                    used_atom_indices.update(chain_a_atom_index_map.get(r, []))

            used_atom_indices = sorted(list(used_atom_indices))
            if not used_atom_indices:
                print("Skipping (no atoms found).")
                continue

            atom_index_to_sliced_pos = {atom_idx: pos for pos, atom_idx in enumerate(used_atom_indices)}
            chunk_sliced = chunk.atom_slice(used_atom_indices)
            coords = chunk_sliced.xyz

            # Map global residue atom indices to the new sliced local indices
            chain_a_positions_map = {resid: [atom_index_to_sliced_pos[a] for a in chain_a_atom_index_map[resid] if a in atom_index_to_sliced_pos]
                                     for resid in chain_a_resids_local}
            chain_b_positions_map = {resid: [atom_index_to_sliced_pos[a] for a in chain_b_atom_index_map[resid] if a in atom_index_to_sliced_pos]
                                     for resid in chain_b_resids_local}

            for pair_key in pairs_to_compute:
                chain_label, resid_i, partner_resid = pair_key
                if chain_label == 'A':
                    source_idxs = chain_a_positions_map.get(resid_i, [])
                    partner_idxs = chain_b_positions_map.get(partner_resid, [])
                else:
                    source_idxs = chain_b_positions_map.get(resid_i, [])
                    partner_idxs = chain_a_positions_map.get(partner_resid, [])

                if len(source_idxs) == 0 or len(partner_idxs) == 0:
                    distance_accumulator[pair_key].append(np.full(n_frames_chunk, np.nan, dtype=np.float32))
                    continue

                coords_i = coords[:, source_idxs, :]
                coords_j = coords[:, partner_idxs, :]

                # Vectorized distance calculation
                diff = coords_i[:, :, None, :] - coords_j[:, None, :, :]
                distance_squared = np.sum(diff**2, axis=-1)
                
                # Reshape to (n_frames, n_atoms_i * n_atoms_j) then take min
                min_distance_squared_per_frame = np.min(distance_squared.reshape(distance_squared.shape[0], -1), axis=1)

                distance_accumulator[pair_key].append(np.sqrt(min_distance_squared_per_frame).astype(np.float32))

            # Free memory explicitly
            del chunk_sliced, coords, coords_i, coords_j, diff, distance_squared
            print("done.")

        print(f"Replica [{replica_num}] INFO - Total frames processed: {total_frames}")

        if total_frames == 0:
            print(f"Replica [{replica_num}] WARNING - No frames processed. Skipping output generation.")
            continue

        window_masks = generate_window_masks(total_frames, NUM_TIME_WINDOWS)
        results_rows = []

        for pair_key in pairs_to_compute:
            chain_label, resid_i, partner_resid = pair_key
            
            if pair_key not in distance_accumulator or not distance_accumulator[pair_key]:
                continue

            distance_array = np.concatenate(distance_accumulator[pair_key]).astype(np.float32)
            valid_mask = ~np.isnan(distance_array)

            if np.any(valid_mask):
                mean_distance = float(np.nanmean(distance_array))
                median_distance = float(np.nanmedian(distance_array))
                std_dev_distance = float(np.nanstd(distance_array))
                occupancy_total = float(np.sum(distance_array[valid_mask] <= DISTANCE_THRESHOLD_NM) / np.sum(valid_mask))
            else:
                mean_distance, median_distance, std_dev_distance, occupancy_total = np.nan, np.nan, np.nan, np.nan

            occupancy_by_window = {}
            for win_name, mask in window_masks.items():
                mask_valid = mask & valid_mask
                if np.sum(mask_valid) == 0:
                    occ = np.nan
                else:
                    occ = float(np.sum(distance_array[mask_valid] <= DISTANCE_THRESHOLD_NM) / np.sum(mask_valid))
                occupancy_by_window[win_name] = occ

            residue_obj = topology_traj.topology.residue(resid_i)
            partner_residue_obj = topology_traj.topology.residue(partner_resid)
            residue_resseq = residue_obj.resSeq
            partner_residue_resseq = partner_residue_obj.resSeq
            partner_chain_label = 'B' if chain_label == 'A' else 'A'

            results_rows.append({
                'chain': chain_label, 
                'residue_mdtraj_idx': resid_i, 
                'residue_pdb_resseq': residue_resseq, 
                'resname': residue_obj.name,
                'chain_partner': partner_chain_label, 
                'partner_residue_mdtraj_idx': partner_resid, 
                'partner_residue_pdb_resseq': partner_residue_resseq, 
                'partner_resname': partner_residue_obj.name,
                'mean_distance_nm': mean_distance, 
                'median_distance_nm': median_distance, 
                'sd_nm': std_dev_distance,
                'occupancy_total': occupancy_total,
                **occupancy_by_window
            })

        df_all_data = pd.DataFrame(results_rows)
        
        if df_all_data.empty:
            print(f"Replica [{replica_num}] WARNING - No data rows generated.")
            continue

        column_order = [
            'chain', 'residue_mdtraj_idx', 'residue_pdb_resseq', 'resname',
            'chain_partner', 'partner_residue_mdtraj_idx', 'partner_residue_pdb_resseq', 'partner_resname',
            'mean_distance_nm', 'median_distance_nm', 'sd_nm',
            'occupancy_total'
        ] + list(window_masks.keys())

        # Ensure we only use columns that exist
        column_order = [c for c in column_order if c in df_all_data.columns]
        df_all_data = df_all_data[column_order]

        def check_high_occupancy(row):
            occupancy_columns = ['occupancy_total'] + list(window_masks.keys())
            occ_values = [row[col] for col in occupancy_columns if col in row]
            # Return True if ANY occupancy value is >= threshold (and not NaN)
            return any((not pd.isna(v)) and v >= OCCUPANCY_FILTER_THRESHOLD for v in occ_values)

        df_filtered_data = df_all_data[df_all_data.apply(check_high_occupancy, axis=1)].copy()

        # --- Save results ---
        try:
            with pd.ExcelWriter(output_xlsx_path, engine='xlsxwriter') as writer:
                df_all_data.to_excel(writer, sheet_name=SHEET_ALL_DATA, index=False)
                df_filtered_data.to_excel(writer, sheet_name=SHEET_FILTERED_DATA, index=False)

            print(f"Replica [{replica_num}] SUCCESS ✅ - Results saved to {output_xlsx_path}")

        except Exception as e:
            csv_out_all = output_xlsx_path.replace('.xlsx', '_all.csv')
            csv_out_filtered = output_xlsx_path.replace('.xlsx', '_filtered.csv')
            df_all_data.to_csv(csv_out_all, index=False)
            df_filtered_data.to_csv(csv_out_filtered, index=False)

            print(f"Replica [{replica_num}] ERROR ❌ - Could not write XLSX: {e}. Saved CSV fallback.")

# Execute the main function when the script runs
if __name__ == "__main__":
    run_trajectory_analysis()